# TAR e SETAR: Modelos com Threshold Abrupto - SOLUTION

Neste notebook exploramos modelos **Threshold Autoregressive (TAR)** e **Self-Exciting TAR (SETAR)**,
que capturam nao-linearidades em series temporais atraves de regimes discretos separados por um
valor limiar (threshold).

## Conteudo
1. Nao-linearidade em series temporais
2. TAR - Threshold Autoregressive (Tong, 1978)
3. SETAR - Self-Exciting TAR
4. Estimacao do threshold $c$ via grid search (Chan, 1993)
5. Teste de Hansen (1996) para significancia do threshold
6. Multiplos regimes - SETAR(3)

## Referencias
- Tong, H. (1978). *On a threshold model*. Pattern Recognition and Signal Processing.
- Tong, H. (1990). *Non-Linear Time Series: A Dynamical System Approach*. Oxford University Press.
- Chan, K.S. (1993). Consistency and limiting distribution of the least squares estimator of a threshold autoregressive model. *Annals of Statistics*, 21, 520-533.
- Hansen, B.E. (1996). Inference when a nuisance parameter is not identified under the null hypothesis. *Econometrica*, 64, 413-430.
- Hansen, B.E. (2000). Sample splitting and threshold estimation. *Econometrica*, 68, 575-603.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from archbox.threshold import (
    SETAR,
    TAR,
    hansen_threshold_test,
)

# Configuracao de graficos
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.style.use('seaborn-v0_8-whitegrid')

# Carregar dados
data = pd.read_csv('../data/us_gdp_growth.csv', parse_dates=['date'], index_col='date')
y = data['y'].values
print(f'US GDP Growth: {len(y)} observacoes')
print(f'Media: {y.mean():.4f}, Desvio-padrao: {y.std():.4f}')
data.head()

## 1. Nao-linearidade em series temporais

Uma forma simples de detectar nao-linearidade e plotar $y_t$ contra $y_{t-1}$.
Se a relacao entre os dois nao for linear (por exemplo, se a dinamica muda
dependendo de $y_{t-1}$ estar acima ou abaixo de um threshold $c$), temos
evidencia de comportamento nao-linear.

O **scatter plot** abaixo permite identificar visualmente se existe um ponto
de corte (threshold) onde a relacao muda.

In [ ]:
# Scatter plot: y_t vs y_{t-1} para identificar visualmente o threshold

y_t = y[1:]      # y_t
y_lag = y[:-1]    # y_{t-1}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot y_t vs y_{t-1}
axes[0].scatter(y_lag, y_t, alpha=0.5, s=20, c='steelblue')
axes[0].set_xlabel(r'$y_{t-1}$', fontsize=12)
axes[0].set_ylabel(r'$y_t$', fontsize=12)
axes[0].set_title(r'Scatter: $y_t$ vs $y_{t-1}$', fontsize=13)
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)

# Threshold candidato visual: mediana
c_visual = np.median(y_lag)
axes[0].axvline(c_visual, color='red', linestyle='--', linewidth=2,
                label=f'Threshold candidato (mediana = {c_visual:.2f})')

# Colorir por regime
regime1 = y_lag <= c_visual
regime2 = y_lag > c_visual
axes[1].scatter(y_lag[regime1], y_t[regime1], alpha=0.5, s=20, c='blue', label='Regime 1 (baixo)')
axes[1].scatter(y_lag[regime2], y_t[regime2], alpha=0.5, s=20, c='red', label='Regime 2 (alto)')
axes[1].axvline(c_visual, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel(r'$y_{t-1}$', fontsize=12)
axes[1].set_ylabel(r'$y_t$', fontsize=12)
axes[1].set_title('Regimes separados pelo threshold', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/scatter_yt_vs_ylag.png', bbox_inches='tight')
plt.show()

## 2. TAR - Threshold Autoregressive

O modelo **TAR** (Tong, 1978) assume que a serie segue diferentes processos AR
dependendo de uma variavel de threshold $s_t$:

$$
y_t = \begin{cases}
\phi_0^{(1)} + \phi_1^{(1)} y_{t-1} + \varepsilon_t^{(1)} & \text{se } s_t \leq c \\
\phi_0^{(2)} + \phi_1^{(2)} y_{t-1} + \varepsilon_t^{(2)} & \text{se } s_t > c
\end{cases}
$$

onde:
- $s_t$ e a **variavel de threshold** (pode ser exogena ou defasagem de $y$)
- $c$ e o **valor do threshold** que separa os regimes
- $\phi^{(j)}$ sao os parametros do regime $j$
- $\varepsilon_t^{(j)} \sim N(0, \sigma_j^2)$ (variancia pode diferir entre regimes)

A transicao e **abrupta**: a serie muda instantaneamente de regime quando $s_t$ cruza $c$.

In [ ]:
# Estimacao TAR com archbox - threshold estimado por grid search

# TAR com variavel de threshold exogena (y_{t-1})
model_tar = TAR(y, order=1, delay=1, n_regimes=2, grid_points=300)
results_tar = model_tar.fit()

print(results_tar.summary())
print(f'\nThreshold estimado: {results_tar.threshold:.4f}')
print(f'AIC: {results_tar.aic:.4f}')
print(f'BIC: {results_tar.bic:.4f}')

## 3. SETAR - Self-Exciting TAR

O **SETAR** (Self-Exciting Threshold Autoregressive) e um caso especial do TAR
onde a variavel de threshold e a propria serie defasada: $s_t = y_{t-d}$.

$$
y_t = \begin{cases}
\phi_0^{(1)} + \phi_1^{(1)} y_{t-1} + \varepsilon_t^{(1)} & \text{se } y_{t-d} \leq c \\
\phi_0^{(2)} + \phi_1^{(2)} y_{t-1} + \varepsilon_t^{(2)} & \text{se } y_{t-d} > c
\end{cases}
$$

O parametro $d$ e o **delay** (defasagem da variavel de transicao).
Um **SETAR(2,1,1)** tem 2 regimes, ordem AR 1 em cada regime, e delay $d=1$.

A escolha do delay $d$ pode ser feita por criterio de informacao (AIC/BIC).

In [ ]:
# Estimacao SETAR(2,1,1) com delay d=1

model_setar = SETAR(y, order=1, delay=1, n_regimes=2, grid_points=300)
results_setar = model_setar.fit()

print(results_setar.summary())
print(f'\nThreshold estimado: {results_setar.threshold:.4f}')
print(f'Delay: {results_setar.delay}')
print(f'\nParametros Regime 1: {results_setar.params_regime1}')
print(f'Parametros Regime 2: {results_setar.params_regime2}')
print(f'\nSigma2 Regime 1: {results_setar.sigma2}')

## 4. Estimacao do threshold $c$

O metodo de **Chan (1993)** estima $c$ por **grid search**: para cada valor candidato
de $c$, calcula-se a soma dos residuos quadrados (RSS) condicional. O threshold
otimo e o que minimiza o RSS.

$$
\hat{c} = \arg\min_c \text{RSS}(c) = \arg\min_c \sum_{t=1}^{T} \hat{\varepsilon}_t^2(c)
$$

O grid e tipicamente construido sobre os quantis da variavel de threshold,
excluindo os extremos (por exemplo, entre o quantil 15% e 85%) para garantir
observacoes suficientes em cada regime.

In [ ]:
# RSS (soma dos residuos quadrados) vs c candidatos - Grid Search de Chan (1993)

from archbox.threshold.base import ThresholdModel

# Grid search manual para visualizar RSS vs c
y_mat, X_mat, s_vec = model_setar._build_matrices()

# Gerar grid de candidatos entre quantis 15% e 85%
c_candidates = np.linspace(np.quantile(s_vec, 0.15), np.quantile(s_vec, 0.85), 200)
rss_values = []

for c in c_candidates:
    mask1 = s_vec <= c
    mask2 = s_vec > c
    if mask1.sum() < 5 or mask2.sum() < 5:
        rss_values.append(np.nan)
        continue
    rss1 = ThresholdModel._ols_rss(y_mat[mask1], X_mat[mask1])
    rss2 = ThresholdModel._ols_rss(y_mat[mask2], X_mat[mask2])
    rss_values.append(rss1 + rss2)

rss_values = np.array(rss_values)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(c_candidates, rss_values, 'b-', linewidth=1.5)
c_opt = c_candidates[np.nanargmin(rss_values)]
ax.axvline(c_opt, color='red', linestyle='--', linewidth=2,
           label=f'Threshold otimo: c = {c_opt:.4f}')
ax.axvline(results_setar.threshold, color='green', linestyle=':', linewidth=2,
           label=f'SETAR estimado: c = {results_setar.threshold:.4f}')
ax.set_xlabel('Threshold candidato $c$', fontsize=12)
ax.set_ylabel('RSS(c)', fontsize=12)
ax.set_title('Grid Search: RSS vs Threshold candidato (Chan, 1993)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/rss_vs_threshold.png', bbox_inches='tight')
plt.show()

## 5. Teste de Hansen (1996)

O teste de **Hansen (1996)** verifica a significancia do threshold.
As hipoteses sao:

- $H_0$: Nao ha efeito de threshold (modelo linear AR)
- $H_1$: Existe threshold (modelo TAR/SETAR)

O problema e que sob $H_0$ o threshold $c$ nao esta identificado (parametro
nuisance nao identificado), o que invalida a distribuicao assintotica usual.
Hansen resolve isso usando um **teste sup-LM** com p-valor obtido por **bootstrap**.

$$
\text{sup-LM} = \sup_c \text{LM}(c)
$$

O p-valor e calculado comparando a estatistica observada com a distribuicao
empirica obtida por $B$ replicas bootstrap sob $H_0$.

In [ ]:
# Teste de Hansen para existencia de threshold

np.random.seed(42)
hansen_result = hansen_threshold_test(y, order=1, delay=1, n_bootstrap=1000, seed=42)

print('=== Teste de Hansen (1996) ===')
print(f'Estatistica sup-LM: {hansen_result.statistic:.4f}')
print(f'P-valor (bootstrap): {hansen_result.pvalue:.4f}')
print(f'Teste: {hansen_result.test_name}')
print(f'Detalhe: {hansen_result.detail}')
print()
if hansen_result.pvalue < 0.05:
    print('=> Rejeitamos H0 a 5%: ha evidencia de efeito threshold.')
else:
    print('=> Nao rejeitamos H0 a 5%: sem evidencia forte de threshold.')

## 6. Multiplos regimes - SETAR(3)

Podemos estender o SETAR para 3 regimes com 2 thresholds $c_1 < c_2$:

$$
y_t = \begin{cases}
\phi_0^{(1)} + \phi_1^{(1)} y_{t-1} + \varepsilon_t & \text{se } y_{t-d} \leq c_1 \\
\phi_0^{(2)} + \phi_1^{(2)} y_{t-1} + \varepsilon_t & \text{se } c_1 < y_{t-d} \leq c_2 \\
\phi_0^{(3)} + \phi_1^{(3)} y_{t-1} + \varepsilon_t & \text{se } y_{t-d} > c_2
\end{cases}
$$

Isso permite capturar comportamentos distintos em recessao, crescimento moderado
e boom economico.

In [ ]:
# SETAR com 3 regimes e 2 thresholds

model_setar3 = SETAR(y, order=1, delay=1, n_regimes=3, grid_points=200)
results_setar3 = model_setar3.fit()

print(results_setar3.summary())
print(f'\nThresholds estimados: {results_setar3.threshold}')
print(f'AIC (2 regimes): {results_setar.aic:.4f}')
print(f'AIC (3 regimes): {results_setar3.aic:.4f}')
print(f'BIC (2 regimes): {results_setar.bic:.4f}')
print(f'BIC (3 regimes): {results_setar3.bic:.4f}')

## Visualizacao dos regimes

Vamos visualizar a serie temporal com os regimes estimados e as linhas de threshold
no scatter plot $y_t$ vs $y_{t-1}$.

In [ ]:
# Scatter plot com linhas de threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SETAR(2) - scatter com threshold
colors = ['blue' if r == 0 else 'red' for r in results_setar.regime_assignments[1:]]
axes[0].scatter(y_lag[:len(colors)], y_t[:len(colors)], c=colors, alpha=0.5, s=20)
axes[0].axvline(results_setar.threshold, color='black', linestyle='--', linewidth=2,
                label=f'c = {results_setar.threshold:.3f}')
axes[0].set_xlabel(r'$y_{t-1}$', fontsize=12)
axes[0].set_ylabel(r'$y_t$', fontsize=12)
axes[0].set_title('SETAR(2): 2 regimes', fontsize=13)
axes[0].legend(fontsize=11)

# SETAR(3) - scatter com 2 thresholds
regime_colors = {0: 'blue', 1: 'green', 2: 'red'}
colors3 = [regime_colors.get(r, 'gray') for r in results_setar3.regime_assignments[1:]]
axes[1].scatter(y_lag[:len(colors3)], y_t[:len(colors3)], c=colors3, alpha=0.5, s=20)
if isinstance(results_setar3.threshold, list):
    for i, c_val in enumerate(results_setar3.threshold):
        axes[1].axvline(c_val, color='black', linestyle='--', linewidth=2,
                        label=f'c{i+1} = {c_val:.3f}')
else:
    axes[1].axvline(results_setar3.threshold, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel(r'$y_{t-1}$', fontsize=12)
axes[1].set_ylabel(r'$y_t$', fontsize=12)
axes[1].set_title('SETAR(3): 3 regimes', fontsize=13)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/scatter_with_thresholds.png', bbox_inches='tight')
plt.show()

## Resumo

| Modelo | Threshold | Transicao | Variavel de transicao |
|--------|-----------|-----------|----------------------|
| TAR    | Estimado  | Abrupta   | Exogena ou $y_{t-d}$ |
| SETAR  | Estimado  | Abrupta   | $y_{t-d}$ (self-exciting) |

**Pontos-chave:**
- O threshold $c$ e estimado por grid search minimizando RSS (Chan, 1993)
- O teste de Hansen (1996) usa bootstrap para testar $H_0$: sem threshold
- SETAR com 3 regimes pode capturar recessao/normal/boom
- A escolha entre 2 e 3 regimes deve ser guiada por AIC/BIC e interpretabilidade

No proximo notebook, veremos modelos com transicao **suave** (LSTAR/ESTAR).